# MobileNetV3-Large

This standalone notebook trains **MobileNetV3-Large**, evaluates the float model, exports dynamic-range and full INT8 TFLite models, benchmarks CPU inference, and archives outputs.

## Download and validate the Kaggle dataset

This cell downloads the prepared dataset into the Colab runtime and checks that train, val, and test contain matching class folders.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile
import shutil

from google.colab import files  # pyright: ignore[reportMissingImports]

KAGGLE_DATASET_ID = "mdhasibultamim/multi-crop-leaf-disease"
DATA_ROOT = Path("/content/leaf_data")
DOWNLOAD_DIR = Path("/content/kaggle_dataset_download")
EXTRACT_DIR = DATA_ROOT / "kaggle_extract"
IMAGE_SUFFIXES = {".bmp", ".gif", ".jpeg", ".jpg", ".png", ".webp", ".tif", ".tiff"}
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

kaggle_config_dir = Path.home() / ".kaggle"
kaggle_config = kaggle_config_dir / "kaggle.json"
if not kaggle_config.is_file():
    print("Upload kaggle.json from Kaggle Account → Create New API Token.")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Please upload a file named kaggle.json")
    kaggle_config_dir.mkdir(parents=True, exist_ok=True)
    kaggle_config.write_bytes(uploaded["kaggle.json"])
os.chmod(kaggle_config, 0o600)
os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_config_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
download = subprocess.run(
    ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET_ID, "-p", str(DOWNLOAD_DIR), "--force"],
    text=True,
    capture_output=True,
)
if download.stdout:
    print(download.stdout)
if download.stderr:
    print(download.stderr)
if download.returncode:
    raise RuntimeError(f"Kaggle download failed for {KAGGLE_DATASET_ID}; verify the API token and dataset access.")

archives = sorted(DOWNLOAD_DIR.glob("*.zip"), key=lambda path: path.stat().st_mtime)
if not archives:
    raise FileNotFoundError(f"No Kaggle archive was downloaded into {DOWNLOAD_DIR}")
with zipfile.ZipFile(archives[-1]) as archive:
    archive.extractall(EXTRACT_DIR)

def inspect_split_tree(root):
    split_classes = {}
    for split in ("train", "val", "test"):
        split_path = root / split
        if not split_path.is_dir():
            return None
        classes = sorted(path.name for path in split_path.iterdir() if path.is_dir())
        if not classes:
            return None
        if not any(path.suffix.lower() in IMAGE_SUFFIXES for path in split_path.rglob("*") if path.is_file()):
            return None
        split_classes[split] = classes
    if any(classes != split_classes["train"] for classes in split_classes.values()):
        raise ValueError(
            "Kaggle train/val/test class folders differ: "
            + ", ".join(f"{split}={len(classes)}" for split, classes in split_classes.items())
        )
    return split_classes["train"]

candidates = [EXTRACT_DIR] + [path for path in EXTRACT_DIR.rglob("processed") if path.is_dir()]
valid_trees = [(path, inspect_split_tree(path)) for path in candidates]
valid_trees = [(path, names) for path, names in valid_trees if names]
if len(valid_trees) != 1:
    raise RuntimeError(
        "Kaggle archive must contain exactly one prepared train/val/test dataset. "
        f"Found {len(valid_trees)} valid split trees under {EXTRACT_DIR}."
    )
source_dataset = valid_trees[0][0]
DATASET_PATH = DATA_ROOT / "processed"
if source_dataset.resolve() != DATASET_PATH.resolve():
    if DATASET_PATH.exists():
        shutil.rmtree(DATASET_PATH)
    shutil.copytree(source_dataset, DATASET_PATH)
DATASET_BASE = str(DATASET_PATH)
class_names = valid_trees[0][1]
print("Using dataset:", DATASET_BASE)
print("Split classes: train=val=test:", len(class_names))
_benchmark_info_path = DATASET_PATH / "dataset_info.json"
if _benchmark_info_path.is_file():
    import json as _json

    with _benchmark_info_path.open(encoding="utf-8") as _handle:
        _info = _json.load(_handle)
    print(
        "Benchmark: cap per class =", _info.get("max_per_class"),
        "| exact duplicates removed =",
        _info.get("removed_exact_duplicates", "n/a"),
    )
    _counts = _info.get("counts") or {}
    for _split in ("train", "val", "test"):
        if _split in _counts:
            print(f"{_split}: {_counts[_split]['total']} images")
    _degenerate = _info.get("degenerate_classes") or {}
    if _degenerate:
        print("Degenerate classes (few unique source images):", _degenerate)
elif (DATASET_PATH / "split_manifest.csv").is_file():
    print("Manifest present but dataset_info.json missing from the archive.")
else:
    print(
        "WARNING: no dataset metadata in this Kaggle archive. Strict split "
        "validation will flag the disclosed degenerate Guava copies as "
        "leakage. Re-upload the benchmark including split_manifest.csv, "
        "dataset_info.json, class_names.json, taxonomy.json, and "
        "near_duplicate_audit.json at the dataset root."
    )


## Clone the project repository


In [ ]:
import os
import subprocess

from google.colab import drive  # pyright: ignore[reportMissingImports]

ARCH = "mobilenet_v3_large"
REPO_URL = "https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git"
REPO_DIR = "/content/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
OUTPUT_BASE = f"/content/drive/MyDrive/leaf_models/{ARCH}"

drive.mount("/content/drive")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
required_paths = ["requirements.txt", "training/train.py", "training/evaluate.py"]
missing_paths = [path for path in required_paths if not os.path.isfile(os.path.join(REPO_DIR, path))]
if missing_paths:
    raise RuntimeError(f"Repository clone is incomplete; missing: {missing_paths}")
os.makedirs(OUTPUT_BASE, exist_ok=True)
print("Repository:", REPO_DIR)
print("Model outputs:", OUTPUT_BASE)


## Install project dependencies

In [ ]:
import os
import subprocess
import sys

requirements_path = os.path.join(REPO_DIR, "requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path], check=True)
print("Dependencies installed from:", requirements_path)

## Configure paths and detect classes

In [ ]:
import os
import sys
import yaml
sys.path.insert(0, REPO_DIR)
from training.runtime_profiles import apply_runtime_profile
RUN_MODE = os.environ.get('TRAINING_RUN_MODE', 'smoke').strip().lower()

cfg_path = os.path.join(REPO_DIR, "training", f"config_{ARCH}.yaml")
with open(cfg_path, "r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

cfg["dataset"]["data_dir"] = DATASET_BASE
for split in ("train", "val", "test"):
    cfg["dataset"][f"{split}_dir"] = os.path.join(DATASET_BASE, split)
cfg["model"]["num_classes"] = len(class_names)
cfg["dataset"]["cache_mode"] = "disk"
cfg["dataset"]["cache_memory_limit_gb"] = 8.0
cfg.setdefault("performance", {}).update({
    "mixed_precision": True,
    "steps_per_execution": 32,
    "jit_compile": False,
    "deterministic": False,
    "distribution": "auto",
    "log_throughput": True,
})
# Training recipe (epochs, unfreeze point, fine-tune LR, cosine schedule,
# loss) is governed entirely by the cloned repo's architecture config
# (Recipe v2: 30 epochs = 20 frozen + 10 fine-tune at LR 2e-5,
# categorical cross-entropy with label smoothing 0.1). Hosted notebooks
# must run the same recipe as Kaggle so results stay comparable.
cfg["callbacks"]["tensorboard"]["enabled"] = False
cfg["export"]["save_dir"] = os.path.dirname(OUTPUT_BASE)
cfg["callbacks"]["csv_logger"]["filename"] = os.path.join(OUTPUT_BASE, f"training_log_{ARCH}.csv")
cfg = apply_runtime_profile(cfg, RUN_MODE, platform='colab')

# Split validation: enabled with a manifest (finishes in seconds, honest
# leakage-free counts); disabled when the Kaggle archive lacks
# split_manifest.csv, whose exact-duplicate scan would hash every image
# (10-25 silent minutes) and flag the disclosed degenerate Guava copies
# as leakage.
manifest_backed = os.path.isfile(os.path.join(DATASET_BASE, "split_manifest.csv"))
if manifest_backed:
    cfg["performance"]["validate_dataset_splits"] = True
    cfg["performance"]["allow_cross_split_duplicates"] = True
else:
    cfg["performance"]["validate_dataset_splits"] = False
    cfg["performance"].pop("allow_cross_split_duplicates", None)
    print(
        "WARNING: split_manifest.csv is missing from the Kaggle archive; deep split "
        "validation is disabled for this run (hashing ~32k images would take 10-25 "
        "minutes and the disclosed degenerate Guava copies would abort training). "
        "Upload dataset/processed.zip (which includes the manifest) as a new "
        "version of the Kaggle dataset to restore manifest-backed validation."
    )
print("Run mode:", RUN_MODE, "| Split validation:",
      "manifest-backed" if manifest_backed else "disabled")
cfg["training"]["log_mode"] = "progress"
# Hosted streams render "\r" in place, so training repaints one
# live status line instead of appending a line per batch.
cfg["training"]["progress_style"] = "inline"
os.makedirs(OUTPUT_BASE, exist_ok=True)
with open(cfg_path, "w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

print(f"{ARCH}: {len(class_names)} classes")
print("Config:", cfg_path)

## Train the model

In [ ]:
# Colab's kernel does not relay OS-level child output to the notebook UI, so
# the training subprocess is streamed through sys.stdout as raw chunks. Chunks
# (not lines) are required: train.py repaints a single in-place progress line
# with "\r", and line-based reads would buffer it until the end of the epoch.
import os
import subprocess
import sys

train_env = os.environ.copy()
train_env["PYTHONUNBUFFERED"] = "1"
train_env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
# Keep TensorFlow's CUDA/XLA INFO lines out of the progress area.
train_env["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("Starting training subprocess; live output will stream below...", flush=True)
training_process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        os.path.join(REPO_DIR, "training", "train.py"),
        "--config",
        cfg_path,
    ],
    env=train_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)
training_output = []
assert training_process.stdout is not None
try:
    while True:
        output_chunk = training_process.stdout.read(4096)
        if not output_chunk:
            break
        output_text = output_chunk.decode("utf-8", errors="replace")
        print(output_text, end="", flush=True)
        training_output.append(output_text)
except KeyboardInterrupt:
    training_process.terminate()
    training_process.wait()
    print("\nInterrupted; terminated the training subprocess.", flush=True)
    raise
training_process.stdout.close()
training_returncode = training_process.wait()
if training_returncode != 0:
    raise RuntimeError(f"Training failed with exit code {training_returncode}")


## Evaluate the float model

In [ ]:
from pathlib import Path

arch_dir = Path(OUTPUT_BASE)
saved_models = sorted(arch_dir.glob("saved_model_*"), key=lambda path: path.stat().st_mtime)
h5_models = sorted(arch_dir.glob("*.h5"), key=lambda path: path.stat().st_mtime)
if saved_models:
    model_path = str(saved_models[-1])
elif h5_models:
    model_path = str(h5_models[-1])
else:
    raise FileNotFoundError(f"No exported model found in {OUTPUT_BASE}")

float_report = os.path.join(OUTPUT_BASE, "evaluation_float")
os.makedirs(float_report, exist_ok=True)
subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "evaluate.py"),
    "--model", model_path,
    "--config", cfg_path,
    "--results-dir", float_report,
], check=True)
print("Evaluated:", model_path)

## Export dynamic-range and full INT8 TFLite models

In [ ]:
quant_script = os.path.join(REPO_DIR, "quantization", "post_training_quant.py")
quant_dir = os.path.join(OUTPUT_BASE, "quantized")
os.makedirs(quant_dir, exist_ok=True)

dynamic_path = os.path.join(quant_dir, f"{ARCH}_dynamic.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", dynamic_path,
    "--arch", ARCH,
], check=True)

int8_path = os.path.join(quant_dir, f"{ARCH}_int8.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", int8_path,
    "--representative_data", cfg["dataset"]["train_dir"],
    "--evaluate",
    "--test_data", cfg["dataset"]["test_dir"],
    "--arch", ARCH,
], check=True)
print("Exported:", dynamic_path, int8_path)

## Benchmark and evaluate TFLite models

In [ ]:
import json
import time
import numpy as np
import tensorflow as tf

benchmark_rows = []
for tflite_path in (dynamic_path, int8_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    input_shape = input_details["shape"]
    if input_details["dtype"] == np.uint8:
        dummy = np.random.randint(0, 256, size=input_shape, dtype=np.uint8)
    else:
        dummy = np.random.uniform(-1.0, 1.0, size=input_shape).astype(np.float32)
    for _ in range(10):
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
    times = []
    for _ in range(50):
        start = time.perf_counter()
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
        times.append((time.perf_counter() - start) * 1000.0)
    row = {
        "model": os.path.basename(tflite_path),
        "input_dtype": np.dtype(input_details["dtype"]).name,
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "size_mb": os.path.getsize(tflite_path) / (1024 * 1024),
    }
    benchmark_rows.append(row)
    print(json.dumps(row, indent=2))

with open(os.path.join(OUTPUT_BASE, "tflite_benchmark.json"), "w", encoding="utf-8") as handle:
    json.dump(benchmark_rows, handle, indent=2)

## Archive outputs

In [ ]:
import shutil

archive_base = os.path.join(os.path.dirname(OUTPUT_BASE), f"{ARCH}_outputs")
archive_path = shutil.make_archive(archive_base, "zip", OUTPUT_BASE)
print("Archive:", archive_path)

## Outputs

The model, class labels, logs, float evaluation report, quantized TFLite files, benchmark JSON, and ZIP archive are stored under the configured output directory.